In [3]:
import requests
import pandas as pd

In [4]:
base_url="https://quickstats.nass.usda.gov/api/"

In [135]:
# 获取具体数据,将查询项放到params里面,params是字典,放回类型是dataframe
def get_data(params):
    url=base_url+"api_GET"
    params["key"]="7BF5BA5C-D7DA-356F-8412-A4710CAE3113"
    params["format"]="JSON"
    response = requests.get(url,params=params)
    if response.ok:
        res=response.json() #dict类型
        res_df=pd.DataFrame(res["data"])
        
        # 格式化df列名
        # 删除一些无用的列
        columns_to_drop = ["begin_code","load_time","end_code","congr_district_code"]
        res_df.drop(columns_to_drop, axis=1, inplace=True)
        
        # 重命名一些列使得与网站筛选时的param名称一致
        columes_rename = {
            "source_desc":"Program",
            "sector_desc":"Sector",
            "group_desc":"Group",
            "commodity_desc":"Commodity",
            "statisticcat_desc":"Category",
            "short_desc":"Data Item",
            "domain_desc":"Domain",
            "domaincat_desc":"Domain Category",
            "agg_level_desc":"Geographic Level",
            "state_name":"State",
            "asd_desc":"Ag District",
            "county_name":"County",
            "region_desc":"Region",
            "zip_5":"Zip Code",
            "watershed_desc":"Watershed",
            "freq_desc":"Period Type",
            "reference_period_desc":"Period"
        }
        res_df.rename(columns=columes_rename,inplace=True)
        
        # 设定列的排列顺序
        columes_order = ['Program','year','Period', 'Period Type','week_ending','Sector',
        'county_ansi', 'county_code','County', 
       'asd_code','Ag District', 'prodn_practice_desc',
       'country_code','country_name',  'unit_desc',  'Domain Category',
       'Watershed', 'location_desc', 'Commodity', 'Domain', 'Category','Region', 'util_practice_desc', 
       'watershed_code', 'state_ansi','state_fips_code', 'state_alpha','State', 'Group',
       'Geographic Level', 'Data Item', 'Zip Code', 
        'class_desc', 'Value','CV (%)']
        res_df=res_df[columes_order]
        
        return res_df
    else:
        status = response.status_code
        status_str='Response code ' + str(status) + ': ' + response.json()['error'][0]
        print(status_str)
        return pd.DataFrame()

In [132]:
# 输入列名(Column or Header Name),放回列下面有哪些项可选
def get_par(par):
    url=base_url+"get_param_values"
    param={"param":par}
    param["key"]="7BF5BA5C-D7DA-356F-8412-A4710CAE3113"
    param["format"]="JSON"
    response = requests.get(url,params=param)
    if response.ok:
        res=response.json() #dict类型
        return pd.DataFrame(res)
    else:
        status = response.status_code
        status_str='Response code ' + str(status) + ': ' + response.json()['error'][0]
        print(status_str)
        return pd.DataFrame()

In [46]:
# 获取被查询数据的行数
def get_counts(params):
    url=base_url+"get_counts"
    params["key"]="7BF5BA5C-D7DA-356F-8412-A4710CAE3113"
    params["format"]="JSON"
    response = requests.get(url,params=params)
    if response.ok:
        res=response.json() #dict类型
        return res["count"]
    else:
        print("get_counts请求错误")
        return None

In [50]:
# 开始筛选数据,先查看列名下面的有哪些项
get_par("source_desc") # 需要选择survey

,source_desc
0,CENSUS
1,SURVEY


In [51]:
get_par("sector_desc") # 需要选择animal&products

,sector_desc
0,ANIMALS & PRODUCTS
1,CROPS
2,DEMOGRAPHICS
3,ECONOMICS
4,ENVIRONMENTAL


In [52]:
get_par("group_desc") # 需要选择livestock或者poultry

,group_desc
0,ANIMAL TOTALS
1,AQUACULTURE
2,COMMODITIES
3,CROP TOTALS
4,DAIRY
5,ENERGY
6,EXPENSES
7,FARMS & LAND & ASSETS
8,FIELD CROPS
9,FRUIT & TREE NUTS


In [53]:
get_par("commodity_desc") # 选择livestock 需要下载cattle/goat /Hogs/sheep/
#如果前面选择了poultry 这里选所有

,commodity_desc
0,AG LAND
1,AG SERVICES
2,AG SERVICES & RENT
3,ALCOHOL COPRODUCTS
4,ALMONDS
...,...
451,WOOD CHIPPERS
452,"WOODY ORNAMENTALS & VINES, OTHER"
453,WOOL
454,YAMS


In [62]:
get_par("statisticcat_desc") # 选择slaughtered，inventory

,statisticcat_desc
0,ACCESSIBILITY
1,"ACCESSIBILITY, 5 YEAR AVG"
2,"ACCESSIBILITY, PREVIOUS YEAR"
3,ACTIVE GINS
4,ACTIVITY
...,...
271,"YEARS ON ANY OPERATION, AVG"
272,"YEARS ON PRESENT OPERATION, AVG"
273,"YEARS RENTED TO TENANT, AVG"
274,YIELD


In [63]:
get_par("short_desc") # 选择SLAUGHTER, COMMERCIAL - SLAUGHTERED,MEASURED IN HEAD 
#和SLAUGHTER, ON FARM - SLAUGHTERED, MEASURED IN HEA

,short_desc
0,AG LAND - ACRES
1,AG LAND - NUMBER OF OPERATIONS
2,AG LAND - OPERATIONS WITH TREATED
3,"AG LAND - TREATED, MEASURED IN ACRES"
4,"AG LAND - TREATED, MEASURED IN CUERDAS"
...,...
34952,"YOGURT, NONFAT, HARD, FROZEN - PRODUCTION, MEA..."
34953,"YOGURT, PLAIN & FLAVORED - PRODUCTION, MEASURE..."
34954,"YOGURT, PLAIN & FLAVORED - PRODUCTION, MEASURE..."
34955,"YOGURT, REGULAR & LOWFAT, HARD, FROZEN - PRODU..."


In [64]:
get_par("agg_level_desc") # 选择state

,agg_level_desc
0,AGRICULTURAL DISTRICT
1,AMERICAN INDIAN RESERVATION
2,COUNTY
3,NATIONAL
4,PUERTO RICO & OUTLYING AREAS
5,REGION : MULTI-STATE
6,REGION : SUB-STATE
7,STATE
8,WATERSHED
9,ZIP CODE


In [117]:
years=get_par("year")

In [139]:
# GROUP选择了livestock
params = {
    "source_desc":"SURVEY",
    "sector_desc":"ANIMALS & PRODUCTS",
    "group_desc":"LIVESTOCK",
    "commodity_desc":["CATTLE","GOATS","HOGS","SHEEP"],
    "statisticcat_desc":["SLAUGHTERED","INVENTORY"],
    "short_desc":["CATTLE, CALVES, SLAUGHTER, COMMERCIAL - SLAUGHTERED, MEASURED IN HEAD","CATTLE, CALVES, SLAUGHTER, ON FARM - SLAUGHTERED, MEASURED IN HEAD"],
    "agg_level_desc":"STATE"
}

In [137]:
# 如果大于5万，则分开
for year in range(2015,2024):
    params["year"]=str(year)
    data=get_data(params)
    data.to_csv(str(year)+".csv",index=False)

In [138]:
# 如果小于5万，则一次性取出，用pandas分类
data=get_data(params)




In [143]:
# GROUP选择了poultry
params = {
    "source_desc":"SURVEY",
    "sector_desc":"ANIMALS & PRODUCTS",
    "group_desc":"POULTRY",
    "commodity_desc":["CHICKENS","DUCKS","EGGS","MEAL","POULTRY BY-PRODUCT MEALS",
                     "POULTRY FATS","POULTRY TOTALS","POULTRY, OTHER","TURKEYS"],
    "statisticcat_desc":["SLAUGHTERED","INVENTORY"],
    "agg_level_desc":"STATE"
}

In [144]:
data=get_data(params)


Response code 413: exceeds limit=50000


In [75]:
# 删除一些无用的列
columns_to_drop = ["begin_code","load_time","end_code","congr_district_code"]
data.drop(columns_to_drop, axis=1, inplace=True)

In [76]:
data

,watershed_code,freq_desc,state_name,util_practice_desc,country_code,reference_period_desc,CV (%),county_ansi,class_desc,asd_code,...,county_code,sector_desc,state_fips_code,region_desc,commodity_desc,source_desc,domain_desc,statisticcat_desc,location_desc,watershed_desc
0,00000000,ANNUAL,ARKANSAS,"SLAUGHTER, COMMERCIAL",9000,YEAR,,,CALVES,,...,,ANIMALS & PRODUCTS,05,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,ARKANSAS,
1,00000000,MONTHLY,ARKANSAS,"SLAUGHTER, COMMERCIAL",9000,SEP,,,CALVES,,...,,ANIMALS & PRODUCTS,05,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,ARKANSAS,
2,00000000,ANNUAL,CALIFORNIA,"SLAUGHTER, COMMERCIAL",9000,YEAR,,,CALVES,,...,,ANIMALS & PRODUCTS,06,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,CALIFORNIA,
3,00000000,MONTHLY,CALIFORNIA,"SLAUGHTER, COMMERCIAL",9000,JAN,,,CALVES,,...,,ANIMALS & PRODUCTS,06,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,CALIFORNIA,
4,00000000,MONTHLY,CALIFORNIA,"SLAUGHTER, COMMERCIAL",9000,FEB,,,CALVES,,...,,ANIMALS & PRODUCTS,06,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,CALIFORNIA,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,00000000,MONTHLY,WISCONSIN,"SLAUGHTER, COMMERCIAL",9000,AUG,,,CALVES,,...,,ANIMALS & PRODUCTS,55,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,WISCONSIN,
198,00000000,MONTHLY,WISCONSIN,"SLAUGHTER, COMMERCIAL",9000,SEP,,,CALVES,,...,,ANIMALS & PRODUCTS,55,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,WISCONSIN,
199,00000000,MONTHLY,WISCONSIN,"SLAUGHTER, COMMERCIAL",9000,OCT,,,CALVES,,...,,ANIMALS & PRODUCTS,55,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,WISCONSIN,
200,00000000,MONTHLY,WISCONSIN,"SLAUGHTER, COMMERCIAL",9000,NOV,,,CALVES,,...,,ANIMALS & PRODUCTS,55,,CATTLE,SURVEY,TOTAL,SLAUGHTERED,WISCONSIN,


In [85]:
# 重命名一些列使得与网站筛选时的param名称一致
columes_rename = {
    "source_desc":"Program",
    "sector_desc":"Sector",
    "group_desc":"Group",
    "commodity_desc":"Commodity",
    "statisticcat_desc":"Category",
    "short_desc":"Data Item",
    "domain_desc":"Domain",
    "domaincat_desc":"Domain Category",
    "agg_level_desc":"Geographic Level",
    "state_name":"State",
    "asd_desc":"Ag District",
    "county_name":"County",
    "region_desc":"Region",
    "zip_5":"Zip Code",
    "watershed_desc":"Watershed",
    "freq_desc":"Period Type",
    "reference_period_desc":"Period"
}
data.rename(columns=columes_rename,inplace=True)

In [97]:
data.columns

Index(['Sector', 'county_code', 'Ag District', 'prodn_practice_desc',
       'country_name', 'County', 'unit_desc', 'week_ending', 'Domain Category',
       'Watershed', 'location_desc', 'Commodity', 'Domain', 'Category',
       'Program', 'Region', 'state_fips_code', 'CV (%)', 'Period',
       'country_code', 'util_practice_desc', 'State', 'Period Type',
       'watershed_code', 'state_alpha', 'Group', 'state_ansi',
       'Geographic Level', 'Value', 'Data Item', 'Zip Code', 'year',
       'asd_code', 'class_desc', 'county_ansi'],
      dtype='object')

In [89]:
c=["Program","Domain"]
data[c]

,Program,Domain
0,SURVEY,TOTAL
1,SURVEY,TOTAL
2,SURVEY,TOTAL
3,SURVEY,TOTAL
4,SURVEY,TOTAL
...,...,...
197,SURVEY,TOTAL
198,SURVEY,TOTAL
199,SURVEY,TOTAL
200,SURVEY,TOTAL


In [23]:
import pandas as pd
for year in range(1989,2024):
    data_year = pd.read_csv(f'D:/中科院数据下载/USDAquickstats/动物类/state/poultry/{year}.csv')
    data_update = data_year[data_year["Period Type"]=="ANNUAL"]
    if data_update.shape[0]==0:
        data_update_yearall = pd.DataFrame()
        for _,data_state in data_year.groupby("State") :
            # 按照州分组
            for _,data_item in data_state.groupby("Data Item"):
                # 唯一的一个州下的一个Data_item，每个月份Value累加成为一整年
                data_item['Value'] = data_item['Value'].replace({',': '', ' ': ''}, regex=True)
                data_item['Value'] = pd.to_numeric(data_item['Value'], errors="ignore")  # 将非数字转换为NaN
                data_item.loc[data_item.index[0], "Value"] = data_item["Value"].sum()
                data_item.loc[data_item.index[0], "Period Type"] = "ANNUAL"
                data_item.loc[data_item.index[0], "Period"] = "YEAR"
                
                data_update_yearall = pd.concat([data_update_yearall,data_item.iloc[0:1]],axis=0)
        data_update_yearall.to_csv("USDAquickstats/动物类/state/poultry_standard/"+str(year)+".csv",index=False)
    else:
        data_update.to_csv("USDAquickstats/动物类/state/poultry_standard/"+str(year)+".csv",index=False)

In [13]:
data_item.iloc[0:1]

,Program,year,Period,Period Type,week_ending,Sector,county_ansi,county_code,County,asd_code,...,state_fips_code,state_alpha,State,Group,Geographic Level,Data Item,Zip Code,class_desc,Value,CV (%)
1826,SURVEY,2023,YEAR,ANNUAL,NaN,ANIMALS & PRODUCTS,NaN,NaN,NaN,NaN,...,55,WI,WISCONSIN,POULTRY,STATE,"TURKEYS, YOUNG, SLAUGHTER, FI - SLAUGHTERED, M...",NaN,YOUNG,"11,253,00011,761,00013,400,00012,389,00012,771...",NaN
